# Testing emotion2vec performance on Savee datatet

In [47]:
!pip install -U funasr modelscope

In [48]:
# Importing libraries
import pandas as pd
import os
from funasr import AutoModel
from google.colab import files, drive
import time
import wave
import numpy as np
from scipy.io.wavfile import write
import tempfile

In [49]:
# Mount google drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
# Load dataset from local storage
df = pd.read_parquet('/content/drive/MyDrive/savee.parquet')
print(df.head())

         file                                              audio gender  \
0  DC_a01.wav  {'bytes': b'RIFF\x1e\xc8\x01\x00WAVEfmt \x10\x...   male   
1  DC_a02.wav  {'bytes': b'RIFF\xea\xad\x01\x00WAVEfmt \x10\x...   male   
2  DC_a03.wav  {'bytes': b'RIFF\x94\x03\x01\x00WAVEfmt \x10\x...   male   
3  DC_a04.wav  {'bytes': b'RIFF\xd0T\x01\x00WAVEfmt \x10\x00\...   male   
4  DC_a05.wav  {'bytes': b'RIFF\xe2v\x01\x00WAVEfmt \x10\x00\...   male   

                                       transcription emotion  speaking_rate  \
0  She had her dark suit in greasy wash water all...   anger          11.79   
1    "'Don't ask me to carry an oily rag like that.'   anger          12.22   
2                              Will you tell me why?   anger           7.71   
3      Who authorised the unlimited expense account?   anger          13.21   
4           Destroy every file related to my audits.   anger          11.67   

   pitch_mean  pitch_std       rms  relative_db  
0  169.301239  27.078539

In [52]:
# Testing with a single data point
head = dfEmotion.head(n = 1)
data = (row['audio']['bytes'])
with tempfile.NamedTemporaryFile(suffix = ".wav", mode = "wb") as file:
    # File writing
    start5 = time.perf_counter()
    file.write(data)
    path = file.name
    end5 = time.perf_counter()
    print(f"Time for writing to file : {end - start} \n")

    model_ids = ["iic/emotion2vec_plus_seed", "iic/emotion2vec_plus_base", "iic/emotion2vec_plus_large"]
    for model_id in model_ids:
      print(f"Current model is {model_id} \n")
      # Load model
      start = time.perf_counter()
      model = AutoModel(
        model=model_id,
        hub="hf",  # "ms" or "modelscope" for China mainland users; "hf" or "huggingface" for other overseas users
        disable_update = True
      )
      end = time.perf_counter()
      print(f"time to load model {model_id} is {end - start}")

      # Emotion detection
      start1 = time.perf_counter()
      result = model.generate(path, language = "en", extract_embedding = False)
      end1 = time.perf_counter()
      print(f"\n Time for emotion detection with model {model_id} is {end1 - start1} \n")

      del model

    # Extract labels and scores
    start2 = time.perf_counter()
    labels = np.array(['angry', 'disgusted', 'fearful', 'happy', 'sad', 'surprised', 'neutral'])
    scores = np.array(result[0]['scores'], dtype = float)

    # Concatenate neutral, other and unk scores into one column
    neutral = scores[4] + scores[5] + scores[8]
    scores = np.delete(scores, [4, 5, 8])
    scores = np.append(scores, [neutral])
    end2 = time.perf_counter()
    print(f"Time for post processing : {end2 - start2} \n")

    # Find the maximum score
    start3 = time.perf_counter()
    max_index = np.argmax(scores)
    max_label = labels[max_index]
    end3 = time.perf_counter()
    print(f"time for maximum element detection : {end3 - start3}")

    # Result
    print(f"\n Emotion detected for {emotion} on row {i} is \n {max_label} \n")

    # Flushing
    start4 = time.perf_counter()
    file.flush()
    end4 = time.perf_counter()
    print(f" time for flushing {end4 - start4} \n")

Time for writing to file : 12.91016275199945 

Current model is iic/emotion2vec_plus_seed 

funasr version: 1.3.14.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9

rtf_avg: 0.346: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


 Time for emotion detection with model iic/emotion2vec_plus_seed is 1.0816594429998077 

Current model is iic/emotion2vec_plus_base 

funasr version: 1.3.14.


...


 Time for emotion detection with model iic/emotion2vec_plus_base is 1.293550561000302 

Current model is iic/emotion2vec_plus_large 

funasr version: 1.3.14.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots

rtf_avg: 0.507: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]


 Time for emotion detection with model iic/emotion2vec_plus_large is 1.5829509409995808 

Time for post processing : 0.0001573190002090996 

time for maximum element detection : 2.3232000785355922e-05

 Emotion detected for disgust on row 19 is 
 disgusted 

 time for flushing 0.00010650900003383867 



In [31]:
# Run the model
start = time.perf_counter()
emotions = df['emotion'].unique()
for emotion in emotions:
    print(f"Current emotion is {emotion} \n")
    dfEmotion = df[df['emotion'] == emotion]
    head = dfEmotion.head()
    print(f"Rows for current emotion {emotion} are : \n {head} \n")

    # Peform emotion recognition on each audio using emotion2vec
    emotionsList = []
    for i, row in head.iterrows():      # row is a Series
        # Store the raw binary data into a temporary audio file
        data = (row['audio']['bytes'])
        with tempfile.NamedTemporaryFile(suffix = ".wav", mode = "wb") as file:
            file.write(data)
            path = file.name
            result = model.generate(path, language = "en", extract_embedding = False)
            emotionDetected = result[0]['labels']
            print(f"Emotion detected for {emotion} on row {i} is {emotionDetected} \n")
            emotionsList.append(emotionDetected)
            file.flush()

    # Compare the predicted emotions with the actual for each emotion
    emotionsSeries = pd.Series(emotionsList)
    correct = head['emotion'] == emotionsSeries
    stats = pd.DataFrame([emotionsSeries, head['emotion'], correct])

    # Store descriptive statistics in a CSV file
    stats.to_csv(path_or_buf = f"/content/drive/My Drive/{emotion}.csv")

    # Store the file in local storage
    files.download(f"/content/drive/My Drive/{emotion}.csv")

    # Clear the intermediatery list and series
    emotionsList.clear()
end = time.perf_counter()
print(f"Time to run : {end - start}")

Current emotion is anger 

Rows for current emotion anger are : 
          file                                              audio gender  \
0  DC_a01.wav  {'bytes': b'RIFF\x1e\xc8\x01\x00WAVEfmt \x10\x...   male   
1  DC_a02.wav  {'bytes': b'RIFF\xea\xad\x01\x00WAVEfmt \x10\x...   male   
2  DC_a03.wav  {'bytes': b'RIFF\x94\x03\x01\x00WAVEfmt \x10\x...   male   
3  DC_a04.wav  {'bytes': b'RIFF\xd0T\x01\x00WAVEfmt \x10\x00\...   male   
4  DC_a05.wav  {'bytes': b'RIFF\xe2v\x01\x00WAVEfmt \x10\x00\...   male   

                                       transcription emotion  speaking_rate  \
0  She had her dark suit in greasy wash water all...   anger          11.79   
1    "'Don't ask me to carry an oily rag like that.'   anger          12.22   
2                              Will you tell me why?   anger           7.71   
3      Who authorised the unlimited expense account?   anger          13.21   
4           Destroy every file related to my audits.   anger          11.67   

   pitch

rtf_avg: 0.490: 100%|██████████| 1/1 [00:01<00:00,  1.79s/it]


Emotion detected for anger on row 0 is ['生气/angry', '厌恶/disgusted', '恐惧/fearful', '开心/happy', '中立/neutral', '其他/other', '难过/sad', '吃惊/surprised', '<unk>'] 



rtf_avg: 0.506: 100%|██████████| 1/1 [00:01<00:00,  1.74s/it]


Emotion detected for anger on row 1 is ['生气/angry', '厌恶/disgusted', '恐惧/fearful', '开心/happy', '中立/neutral', '其他/other', '难过/sad', '吃惊/surprised', '<unk>'] 



...

rtf_avg: 0.476: 100%|██████████| 1/1 [00:02<00:00,  2.09s/it]


Emotion detected for disgust on row 18 is ['生气/angry', '厌恶/disgusted', '恐惧/fearful', '开心/happy', '中立/neutral', '其他/other', '难过/sad', '吃惊/surprised', '<unk>'] 



rtf_avg: 0.495: 100%|██████████| 1/1 [00:01<00:00,  1.54s/it]

Emotion detected for disgust on row 19 is ['生气/angry', '厌恶/disgusted', '恐惧/fearful', '开心/happy', '中立/neutral', '其他/other', '难过/sad', '吃惊/surprised', '<unk>'] 



ValueError: Can only compare identically-labeled Series objects